# Exp 3: Token Diversity & Attention Entropy Analysis

Measures whether ViT-5's components (RoPE, registers, QK-Norm) actually improve
spatial modeling and prevent over-smoothing, as claimed.

We measure at every layer:
- **Token cosine similarity** (avg pairwise cos-sim between tokens — higher = more over-smoothed)
- **Attention entropy** (Shannon entropy of attention weights per head — higher = more diffuse)
- **Cross-head diversity** (avg pairwise cos-sim between attention patterns of different heads — lower = more diverse)
- **CLS-to-patch attention concentration** (how focused is the CLS token's attention)

Models: ViT-5-Small vs DeiT-III-Small. Memory-efficient: online computation, T4-safe.

In [ ]:
!pip install -q timm einops huggingface_hub matplotlib

In [ ]:
!git clone https://github.com/wangf3014/ViT-5.git vit5_repo 2>/dev/null || echo 'Already cloned'

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs('checkpoints', exist_ok=True)
vit5_small_ckpt = hf_hub_download(
    repo_id='FengWang3211/ViT-5',
    filename='vit5_small_patch16_224.pth',
    local_dir='checkpoints'
)
print(f'Downloaded: {vit5_small_ckpt}')

In [ ]:
import sys
sys.path.insert(0, 'vit5_repo')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from models_vit5 import vit5_small, deit_small_patch16_LS
import timm

def load_vit5_small():
    model = vit5_small(img_size=224)
    ckpt = torch.load(vit5_small_ckpt, map_location='cpu', weights_only=False)
    state_dict = ckpt['model'] if 'model' in ckpt else ckpt
    model.load_state_dict(state_dict, strict=False)
    return model.to(device).eval()

def load_deit3_small():
    model = timm.create_model('deit3_small_patch16_224.fb_in1k', pretrained=True)
    return model.to(device).eval()

print('Model loaders ready.')

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

IMAGENET_VAL = '/content/imagenet/val'
if os.path.exists(IMAGENET_VAL):
    print('Using ImageNet validation set')
    dataset = datasets.ImageFolder(IMAGENET_VAL, transform=transform)
else:
    print('Using CIFAR-100 as proxy.')
    dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

NUM_SAMPLES = 200  # Attention matrices are large; keep sample count lower
subset = torch.utils.data.Subset(dataset, list(range(min(NUM_SAMPLES, len(dataset)))))
loader = torch.utils.data.DataLoader(subset, batch_size=16, shuffle=False, num_workers=2)
print(f'{len(subset)} samples, batch_size=16, {len(loader)} batches')

In [ ]:
# ============================================================
# Patched attention forward to capture attention weights
# We monkey-patch the Attention.forward to store attn weights
# without flash attention (already flash=False in both models)
# ============================================================

class OnlineTokenDiversityTracker:
    """Computes token diversity and attention entropy metrics online."""
    
    def __init__(self, num_layers, num_heads):
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.reset()
    
    def reset(self):
        # Token diversity (cosine similarity)
        self.token_cossim_sum = [0.0] * self.num_layers
        
        # Attention entropy per head
        self.attn_entropy_sum = [[0.0] * self.num_heads for _ in range(self.num_layers)]
        
        # Cross-head diversity
        self.cross_head_sim_sum = [0.0] * self.num_layers
        
        # CLS attention concentration (entropy of CLS row)
        self.cls_entropy_sum = [[0.0] * self.num_heads for _ in range(self.num_layers)]
        
        self.count = 0
    
    def update_tokens(self, layer_idx, hidden_states):
        """Update token diversity from block output. hidden_states: (B, N, D) on GPU."""
        with torch.no_grad():
            B, N, D = hidden_states.shape
            h = hidden_states.float()
            
            # Normalize tokens
            h_norm = F.normalize(h, dim=-1)  # (B, N, D)
            
            # Pairwise cosine similarity: sample a subset of pairs for efficiency
            # Full: (B, N, N) — too large. Sample 100 random pairs per image.
            n_pairs = min(100, N * (N - 1) // 2)
            idx_i = torch.randint(0, N, (n_pairs,), device=h.device)
            idx_j = torch.randint(0, N, (n_pairs,), device=h.device)
            # Avoid self-pairs
            mask = idx_i != idx_j
            idx_i, idx_j = idx_i[mask], idx_j[mask]
            
            if len(idx_i) > 0:
                cos_sim = (h_norm[:, idx_i] * h_norm[:, idx_j]).sum(dim=-1)  # (B, n_pairs)
                self.token_cossim_sum[layer_idx] += cos_sim.mean().item()
    
    def update_attention(self, layer_idx, attn_weights):
        """Update attention metrics. attn_weights: (B, H, N, N) on GPU."""
        with torch.no_grad():
            B, H, N, _ = attn_weights.shape
            attn = attn_weights.float()
            
            # --- Attention entropy per head ---
            # Shannon entropy: -sum(p * log(p)), computed per query token, averaged
            log_attn = torch.log(attn + 1e-12)
            entropy = -(attn * log_attn).sum(dim=-1)  # (B, H, N)
            entropy_per_head = entropy.mean(dim=(0, 2))  # (H,)
            for h in range(H):
                self.attn_entropy_sum[layer_idx][h] += entropy_per_head[h].item()
            
            # --- CLS token attention entropy (row 0) ---
            cls_attn = attn[:, :, 0, :]  # (B, H, N)
            cls_log = torch.log(cls_attn + 1e-12)
            cls_entropy = -(cls_attn * cls_log).sum(dim=-1)  # (B, H)
            cls_entropy_per_head = cls_entropy.mean(dim=0)  # (H,)
            for h in range(H):
                self.cls_entropy_sum[layer_idx][h] += cls_entropy_per_head[h].item()
            
            # --- Cross-head diversity ---
            # Flatten attention maps per head: (B, H, N*N)
            attn_flat = attn.reshape(B, H, -1)
            attn_norm = F.normalize(attn_flat, dim=-1)  # (B, H, N*N)
            # Pairwise cosine sim between heads: (B, H, H)
            head_sim = torch.bmm(attn_norm, attn_norm.transpose(1, 2))  # (B, H, H)
            # Average off-diagonal elements
            mask = ~torch.eye(H, device=attn.device, dtype=torch.bool).unsqueeze(0)
            off_diag = head_sim[mask.expand(B, -1, -1)].reshape(B, -1)
            self.cross_head_sim_sum[layer_idx] += off_diag.mean().item()
    
    def get_metrics(self):
        n = self.count
        if n == 0:
            return {}
        metrics = {}
        for i in range(self.num_layers):
            metrics[i] = {
                'token_cossim': self.token_cossim_sum[i] / n,
                'attn_entropy_per_head': [s / n for s in self.attn_entropy_sum[i]],
                'attn_entropy_avg': np.mean([s / n for s in self.attn_entropy_sum[i]]),
                'cls_entropy_per_head': [s / n for s in self.cls_entropy_sum[i]],
                'cls_entropy_avg': np.mean([s / n for s in self.cls_entropy_sum[i]]),
                'cross_head_sim': self.cross_head_sim_sum[i] / n,
            }
        return metrics

print('OnlineTokenDiversityTracker ready.')

In [ ]:
# ============================================================
# Analysis function: hooks on block output + attention weights
# ============================================================

def analyze_model_vit5(model, loader, model_name):
    """Analyze ViT-5 model (custom Attention class with extractable attn weights)."""
    num_layers = len(model.blocks)
    num_heads = model.blocks[0].attn.num_heads
    tracker = OnlineTokenDiversityTracker(num_layers, num_heads)
    
    hooks = []
    
    # Hook on block output for token diversity
    for i, block in enumerate(model.blocks):
        def make_block_hook(idx):
            def hook_fn(module, input, output):
                tracker.update_tokens(idx, output)
            return hook_fn
        hooks.append(block.register_forward_hook(make_block_hook(i)))
    
    # Patch attention forward to capture weights
    original_forwards = []
    for i, block in enumerate(model.blocks):
        attn_module = block.attn
        original_forwards.append(attn_module.forward)
        
        def make_patched_forward(orig_fwd, attn_mod, layer_idx):
            def patched_forward(x, **kwargs):
                B, N, C = x.shape
                qkv = attn_mod.qkv(x).reshape(B, N, 3, attn_mod.num_heads, C // attn_mod.num_heads)
                q, k, v = qkv.unbind(dim=2)
                
                if attn_mod.qk_norm:
                    qk_dtype = q.dtype
                    q = attn_mod.q_norm(q).to(qk_dtype)
                    k = attn_mod.k_norm(k).to(qk_dtype)
                
                reg_idx = N - attn_mod.num_registers
                if attn_mod.rope is not None:
                    q = torch.cat((q[:, :1], attn_mod.rope(q[:, 1:reg_idx]), q[:, reg_idx:]), dim=1)
                    k = torch.cat((k[:, :1], attn_mod.rope(k[:, 1:reg_idx]), k[:, reg_idx:]), dim=1)
                if attn_mod.rope_reg is not None:
                    q = torch.cat((q[:, :1], q[:, 1:reg_idx], attn_mod.rope_reg(q[:, reg_idx:])), dim=1)
                    k = torch.cat((k[:, :1], k[:, 1:reg_idx], attn_mod.rope_reg(k[:, reg_idx:])), dim=1)
                
                q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
                attn_weights = (q * attn_mod.scale) @ k.transpose(-2, -1)
                attn_weights = attn_weights.softmax(dim=-1)
                attn_weights = attn_mod.attn_drop(attn_weights)
                
                # Store and use attention weights
                tracker.update_attention(layer_idx, attn_weights)
                
                x = (attn_weights @ v).transpose(1, 2).reshape(B, N, C)
                x = attn_mod.proj(x)
                x = attn_mod.proj_drop(x)
                return x
            return patched_forward
        
        attn_module.forward = make_patched_forward(
            attn_module.forward, attn_module, i
        )
    
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):
            images = images.to(device)
            _ = model(images)
            tracker.count += 1
            if (batch_idx + 1) % 4 == 0:
                print(f'  [{model_name}] {batch_idx+1}/{len(loader)}')
    
    # Restore original forwards
    for i, block in enumerate(model.blocks):
        block.attn.forward = original_forwards[i]
    for h in hooks:
        h.remove()
    
    return tracker.get_metrics()


def analyze_model_deit(model, loader, model_name):
    """Analyze DeiT-III model (timm's Attention with standard structure)."""
    num_layers = len(model.blocks)
    num_heads = model.blocks[0].attn.num_heads
    tracker = OnlineTokenDiversityTracker(num_layers, num_heads)
    hooks = []
    
    # Hook on block output for token diversity
    for i, block in enumerate(model.blocks):
        def make_block_hook(idx):
            def hook_fn(module, input, output):
                tracker.update_tokens(idx, output)
            return hook_fn
        hooks.append(block.register_forward_hook(make_block_hook(i)))
    
    # Patch attention to extract weights
    original_forwards = []
    for i, block in enumerate(model.blocks):
        attn_module = block.attn
        original_forwards.append(attn_module.forward)
        
        def make_patched_forward(attn_mod, layer_idx):
            def patched_forward(x, **kwargs):
                B, N, C = x.shape
                qkv = attn_mod.qkv(x).reshape(B, N, 3, attn_mod.num_heads, C // attn_mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, v = qkv.unbind(0)
                
                attn_weights = (q @ k.transpose(-2, -1)) * attn_mod.scale
                attn_weights = attn_weights.softmax(dim=-1)
                attn_weights = attn_mod.attn_drop(attn_weights)
                
                tracker.update_attention(layer_idx, attn_weights)
                
                x = (attn_weights @ v).transpose(1, 2).reshape(B, N, C)
                x = attn_mod.proj(x)
                x = attn_mod.proj_drop(x)
                return x
            return patched_forward
        
        attn_module.forward = make_patched_forward(attn_module, i)
    
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):
            images = images.to(device)
            _ = model(images)
            tracker.count += 1
            if (batch_idx + 1) % 4 == 0:
                print(f'  [{model_name}] {batch_idx+1}/{len(loader)}')
    
    for i, block in enumerate(model.blocks):
        block.attn.forward = original_forwards[i]
    for h in hooks:
        h.remove()
    
    return tracker.get_metrics()

print('Analysis functions ready.')

In [ ]:
# ============================================================
# Run ViT-5-Small
# ============================================================

print('Loading ViT-5-Small...')
model_vit5 = load_vit5_small()
print(f'  {sum(p.numel() for p in model_vit5.parameters())/1e6:.1f}M params')

print('Analyzing ViT-5-Small...')
metrics_vit5 = analyze_model_vit5(model_vit5, loader, 'ViT-5-Small')

del model_vit5
torch.cuda.empty_cache()
print('ViT-5-Small done, GPU freed.')

In [ ]:
# ============================================================
# Run DeiT-III-Small
# ============================================================

print('Loading DeiT-III-Small...')
model_deit = load_deit3_small()
print(f'  {sum(p.numel() for p in model_deit.parameters())/1e6:.1f}M params')

print('Analyzing DeiT-III-Small...')
metrics_deit = analyze_model_deit(model_deit, loader, 'DeiT-III-Small')

del model_deit
torch.cuda.empty_cache()
print('DeiT-III-Small done, GPU freed.')

In [ ]:
# ============================================================
# Print numerical results
# ============================================================

num_layers = len(metrics_vit5)

print('TOKEN DIVERSITY & ATTENTION ENTROPY')
print('=' * 100)
print(f"{'Layer':>5} | {'TokenSim(V5)':>13} {'TokenSim(D3)':>13} {'diff':>7} | "
      f"{'AttnEnt(V5)':>12} {'AttnEnt(D3)':>12} {'diff':>7} | "
      f"{'HeadSim(V5)':>12} {'HeadSim(D3)':>12}")
print('-' * 100)

for layer in range(num_layers):
    v5 = metrics_vit5[layer]
    d3 = metrics_deit[layer]
    print(f"{layer:>5} | {v5['token_cossim']:>13.4f} {d3['token_cossim']:>13.4f} {v5['token_cossim']-d3['token_cossim']:>+7.4f} | "
          f"{v5['attn_entropy_avg']:>12.4f} {d3['attn_entropy_avg']:>12.4f} {v5['attn_entropy_avg']-d3['attn_entropy_avg']:>+7.4f} | "
          f"{v5['cross_head_sim']:>12.4f} {d3['cross_head_sim']:>12.4f}")

print('-' * 100)
avg_v5_ts = np.mean([metrics_vit5[l]['token_cossim'] for l in range(num_layers)])
avg_d3_ts = np.mean([metrics_deit[l]['token_cossim'] for l in range(num_layers)])
avg_v5_ae = np.mean([metrics_vit5[l]['attn_entropy_avg'] for l in range(num_layers)])
avg_d3_ae = np.mean([metrics_deit[l]['attn_entropy_avg'] for l in range(num_layers)])
avg_v5_hs = np.mean([metrics_vit5[l]['cross_head_sim'] for l in range(num_layers)])
avg_d3_hs = np.mean([metrics_deit[l]['cross_head_sim'] for l in range(num_layers)])

print(f"{'AVG':>5} | {avg_v5_ts:>13.4f} {avg_d3_ts:>13.4f} {avg_v5_ts-avg_d3_ts:>+7.4f} | "
      f"{avg_v5_ae:>12.4f} {avg_d3_ae:>12.4f} {avg_v5_ae-avg_d3_ae:>+7.4f} | "
      f"{avg_v5_hs:>12.4f} {avg_d3_hs:>12.4f}")

In [ ]:
# ============================================================
# CLS token attention entropy
# ============================================================

print('\nCLS TOKEN ATTENTION ENTROPY (how focused is the class token)')
print('=' * 80)
print(f"{'Layer':>5} | {'CLS_Ent(V5)':>12} {'CLS_Ent(D3)':>12} {'diff':>7}")
print('-' * 40)

for layer in range(num_layers):
    v5 = metrics_vit5[layer]['cls_entropy_avg']
    d3 = metrics_deit[layer]['cls_entropy_avg']
    print(f"{layer:>5} | {v5:>12.4f} {d3:>12.4f} {v5-d3:>+7.4f}")

In [ ]:
# ============================================================
# Figure 1: 3-panel comparison across layers
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Token Diversity & Attention Entropy: ViT-5-Small vs DeiT-III-Small', 
             fontsize=14, fontweight='bold')

layers = list(range(num_layers))

# Panel 1: Token cosine similarity (over-smoothing)
ax = axes[0]
ax.plot(layers, [metrics_vit5[l]['token_cossim'] for l in layers], 'o-', color='#e74c3c', lw=2, ms=6, label='ViT-5')
ax.plot(layers, [metrics_deit[l]['token_cossim'] for l in layers], 's--', color='#3498db', lw=2, ms=6, label='DeiT-III')
ax.set_xlabel('Layer')
ax.set_ylabel('Avg Pairwise Cosine Similarity')
ax.set_title('Token Similarity (higher = more over-smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(layers)

# Panel 2: Attention entropy
ax = axes[1]
ax.plot(layers, [metrics_vit5[l]['attn_entropy_avg'] for l in layers], 'o-', color='#e74c3c', lw=2, ms=6, label='ViT-5')
ax.plot(layers, [metrics_deit[l]['attn_entropy_avg'] for l in layers], 's--', color='#3498db', lw=2, ms=6, label='DeiT-III')
ax.set_xlabel('Layer')
ax.set_ylabel('Shannon Entropy (nats)')
ax.set_title('Attention Entropy (higher = more diffuse)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(layers)

# Panel 3: Cross-head similarity
ax = axes[2]
ax.plot(layers, [metrics_vit5[l]['cross_head_sim'] for l in layers], 'o-', color='#e74c3c', lw=2, ms=6, label='ViT-5')
ax.plot(layers, [metrics_deit[l]['cross_head_sim'] for l in layers], 's--', color='#3498db', lw=2, ms=6, label='DeiT-III')
ax.set_xlabel('Layer')
ax.set_ylabel('Avg Cross-Head Cosine Similarity')
ax.set_title('Head Redundancy (lower = more diverse heads)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(layers)

plt.tight_layout()
plt.savefig('exp3_token_attention_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp3_token_attention_overview.png')

In [ ]:
# ============================================================
# Figure 2: Per-head attention entropy heatmaps
# ============================================================

num_heads = len(metrics_vit5[0]['attn_entropy_per_head'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Attention Entropy per Head per Layer', fontsize=14, fontweight='bold')

for ax, metrics, name in [
    (axes[0], metrics_vit5, 'ViT-5-Small'),
    (axes[1], metrics_deit, 'DeiT-III-Small')
]:
    data = np.array([metrics[l]['attn_entropy_per_head'] for l in range(num_layers)])
    im = ax.imshow(data, aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_xlabel('Head Index')
    ax.set_ylabel('Layer Index')
    ax.set_title(name)
    ax.set_xticks(range(num_heads))
    ax.set_yticks(range(num_layers))
    plt.colorbar(im, ax=ax, label='Entropy (nats)')

plt.tight_layout()
plt.savefig('exp3_attention_entropy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp3_attention_entropy_heatmap.png')

In [ ]:
# ============================================================
# Figure 3: CLS attention entropy
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(layers, [metrics_vit5[l]['cls_entropy_avg'] for l in layers], 'o-', color='#e74c3c', lw=2, ms=6, label='ViT-5')
ax.plot(layers, [metrics_deit[l]['cls_entropy_avg'] for l in layers], 's--', color='#3498db', lw=2, ms=6, label='DeiT-III')
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('CLS Token Attention Entropy (nats)', fontsize=12)
ax.set_title('CLS Token Attention Focus\n(lower entropy = more concentrated/focused attention)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(layers)

plt.tight_layout()
plt.savefig('exp3_cls_attention_entropy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp3_cls_attention_entropy.png')

In [ ]:
# ============================================================
# Summary
# ============================================================

print('=' * 70)
print('SUMMARY: Token Diversity & Attention Entropy')
print('=' * 70)
print()
print('Q1: Does ViT-5 suffer from over-smoothing (tokens becoming similar)?')
print(f'    ViT-5   avg token cos-sim: {avg_v5_ts:.4f}')
print(f'    DeiT-III avg token cos-sim: {avg_d3_ts:.4f}')
diff_ts = avg_v5_ts - avg_d3_ts
print(f'    Delta: {diff_ts:+.4f} ({"ViT-5 more over-smoothed" if diff_ts > 0 else "DeiT-III more over-smoothed"})')
print()
print('Q2: Does ViT-5 have higher attention entropy (more diffuse attention)?')
print(f'    ViT-5   avg attn entropy: {avg_v5_ae:.4f}')
print(f'    DeiT-III avg attn entropy: {avg_d3_ae:.4f}')
diff_ae = avg_v5_ae - avg_d3_ae
print(f'    Delta: {diff_ae:+.4f}')
print()
print('Q3: Do attention heads in ViT-5 learn more diverse patterns?')
print(f'    ViT-5   avg cross-head sim: {avg_v5_hs:.4f}')
print(f'    DeiT-III avg cross-head sim: {avg_d3_hs:.4f}')
diff_hs = avg_v5_hs - avg_d3_hs
print(f'    Delta: {diff_hs:+.4f} ({"ViT-5 heads more redundant" if diff_hs > 0 else "DeiT-III heads more redundant"})')
print()
print('INTERPRETATION:')
print('- Lower token cos-sim = better (more diverse representations, less over-smoothing)')
print('- Attention entropy: context-dependent (focused is good for classification,')
print('  diffuse can be good for spatial tasks)')
print('- Lower cross-head sim = better (heads capture different patterns)')
print()
print('These metrics test whether ViT-5\'s RoPE, registers, and QK-Norm')
print('actually improve spatial modeling as claimed in the paper.')